# **AI Advanced Ahmed Yousrey Course Practice**

---
# **DAY 10 — Vehicle Price Prediction**
---

# **STEP 1 - IMPORT LIBRARIES**

---

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import joblib
import warnings
warnings.filterwarnings("ignore")

## **STEP 2 - LOAD DATA**

---

In [8]:
import pandas as pd

df = pd.read_excel(
    r"D:\ML adv\AI advanced my project\project 10 Vehicle Price Prediction\data\submission.xlsx"
)


## **STEP 3 - EDA**

---

In [18]:
df.shape

(2604, 21)

In [17]:
df.head()

,location,condition,Listed_date,Listed_time,drive,fuel,lat,long,odometer,paint color,...,size,title status,transmission,type,year make model,cylinders,year,make,re_model,MODELS
0,dallas,excellent,2022-06-01,18;17,4wd,gas,32.856174,-96.672722,200396.0,NaN,...,NaN,clean,automatic,SUV,2010 Nissan Xterra,6,2010,Nissan,Nissan Xterra,Xterra
1,dallas,excellent,2022-06-01,18;16,fwd,hybrid,32.856174,-96.672722,152434.0,NaN,...,NaN,clean,automatic,SUV,2007 Lexus RX 400h,6,2007,Lexus,Lexus RX 400h,RX 400h
2,dallas,excellent,2022-06-01,18;16,fwd,gas,32.778500,-97.083400,90000.0,blue,...,mid-size,clean,automatic,sedan,2015 hyundai elantra,4,2015,hyundai,hyundai elantra,elantra
3,dallas,NaN,2022-06-01,18;16,fwd,gas,33.681077,-112.058490,61482.0,NaN,...,NaN,clean,automatic,van,2007 Honda Odyssey EX Power Side-Ent,6,2007,Honda,Honda Odyssey EX Power Side-Ent,Odyssey EX Power Side-Ent
4,dallas,excellent,2022-06-01,18;16,rwd,gas,32.856174,-96.672722,70523.0,NaN,...,mid-size,clean,automatic,sedan,2005 Infiniti G35 Sedan,6,2005,Infiniti,Infiniti G35 Sedan,G35 Sedan


In [16]:
df.isnull().sum()

location              0
condition           969
Listed_date           0
Listed_time           0
drive               728
fuel                  1
lat                  21
long                 21
odometer              8
paint color         732
price                 3
size               1609
title status          0
transmission          2
type                548
year make model       0
cylinders          1020
year                  0
make                  2
re_model              0
MODELS               30
dtype: int64

In [15]:
df.describe()

,Listed_date,lat,long,odometer,price,year
count,2604,2583.000000,2583.000000,2.596000e+03,2601.000000,2604.000000
mean,2022-05-30 06:13:49.493087488,33.160744,-97.764867,1.068846e+05,22561.376394,2010.847158
min,2022-05-27 00:00:00,21.589269,-122.684200,1.000000e+00,1.000000,1932.000000
25%,2022-05-29 00:00:00,32.749367,-97.225295,5.301425e+04,8500.000000,2008.000000
50%,2022-05-30 00:00:00,32.840011,-96.949300,9.200000e+04,18950.000000,2014.000000
75%,2022-05-31 00:00:00,33.007775,-96.760000,1.401238e+05,30995.000000,2017.000000
max,2022-06-01 00:00:00,47.770214,-80.150704,3.333333e+06,299997.000000,2023.000000
std,NaN,1.730026,4.536041,1.135645e+05,19356.100193,10.090241


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2604 entries, 0 to 2603
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   location         2604 non-null   object        
 1   condition        1635 non-null   object        
 2   Listed_date      2604 non-null   datetime64[ns]
 3   Listed_time      2604 non-null   object        
 4   drive            1876 non-null   object        
 5   fuel             2603 non-null   object        
 6   lat              2583 non-null   float64       
 7   long             2583 non-null   float64       
 8   odometer         2596 non-null   float64       
 9   paint color      1872 non-null   object        
 10  price            2601 non-null   float64       
 11  size             995 non-null    object        
 12  title status     2604 non-null   object        
 13  transmission     2602 non-null   object        
 14  type             2056 non-null   object 

## **STEP 4 - PREPROCESSING**

---

In [19]:
# Drop redundant and irrelevant columns
df.drop(columns=["location", "Listed_date", "Listed_time", "lat", "long","year make model", "re_model", "size"], inplace=True)

# Drop rows where price is null (target column)
df.dropna(subset=["price"], inplace=True)

print(df.shape)
print(df.isnull().sum())

(2601, 13)
condition        969
drive            728
fuel               1
odometer           8
paint color      732
price              0
title status       0
transmission       2
type             548
cylinders       1017
year               0
make               2
MODELS            30
dtype: int64


---
handle missing values

In [20]:
# Fill categorical nulls with unknown
for col in ["condition", "drive", "paint color", "type", "cylinders"]:
    df[col].fillna("unknown", inplace=True)

# Drop rows with very few nulls
df.dropna(subset=["fuel", "transmission", "make", "MODELS"], inplace=True)

# Fill odometer with median
df["odometer"].fillna(df["odometer"].median(), inplace=True)

print(df.shape)
print(df.isnull().sum())

(2566, 13)
condition       0
drive           0
fuel            0
odometer        0
paint color     0
price           0
title status    0
transmission    0
type            0
cylinders       0
year            0
make            0
MODELS          0
dtype: int64


In [21]:
print(df["price"].describe())

count      2566.000000
mean      22596.881917
std       19262.095472
min           1.000000
25%        8600.000000
50%       18989.000000
75%       30995.000000
max      299997.000000
Name: price, dtype: float64


min is $1 and max is nearly $300k. Those are bad listings.

Filter to a realistic price range:

In [22]:
df = df[(df["price"] >= 500) & (df["price"] <= 150000)]
print(df.shape)

(2551, 13)


---

In [23]:
print(df["odometer"].describe())

count    2.551000e+03
mean     1.065711e+05
std      1.132298e+05
min      1.000000e+00
25%      5.330500e+04
50%      9.134600e+04
75%      1.400000e+05
max      3.333333e+06
Name: odometer, dtype: float64


Max is 3.3 million miles 

 clearly bad data. Filter it:

In [24]:
df = df[(df["odometer"] >= 1000) & (df["odometer"] <= 500000)]
print(df.shape)

(2466, 13)


---

encode the categorical columns

In [30]:
cat_cols = ["condition", "drive", "fuel", "paint color", "title status",
            "transmission", "type", "cylinders", "make", "MODELS"]

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print(df.head())

   condition  drive  fuel  odometer  paint color    price  title status  \
0          0      0     2  200396.0            8   6995.0             0   
1          0      1     3  152434.0            8   9495.0             0   
2          0      1     2   90000.0            1    800.0             0   
3          6      1     2   61482.0            8  25900.0             0   
4          0      2     2   70523.0            8   8995.0             0   

   transmission  type  cylinders  year  make  MODELS  
0             0     0          4  2010    42     179  
1             0     0          4  2007    23    1214  
2             0     9          2  2015    80     370  
3             0    12          4  2007   159    1135  
4             0     9          4  2005     7     953  


In [28]:
for col in cat_cols:
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])

print(df.head())

   condition  drive  fuel  odometer  paint color    price  title status  \
0          0      0     2  200396.0            8   6995.0             0   
1          0      1     3  152434.0            8   9495.0             0   
2          0      1     2   90000.0            1    800.0             0   
3          6      1     2   61482.0            8  25900.0             0   
4          0      2     2   70523.0            8   8995.0             0   

   transmission  type  cylinders  year  make  MODELS  
0             0     0          4  2010    42     179  
1             0     0          4  2007    23    1214  
2             0     9          2  2015    80     370  
3             0    12          4  2007   159    1135  
4             0     9          4  2005     7     953  


---
train/test split

In [29]:
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

(1972, 12) (494, 12)


---
## **STEP 5 - Model Training**

---

In [33]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"R2: {r2:.4f}")

MAE: 5583.62
R2: 0.7216


---
## **STEP 6 - Save Model**

---

In [34]:
joblib.dump(model, "app/model.pkl")
print("Model saved.")

Model saved.
